### imports

In [8]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from my_ml.datasets import load_spambase

In [9]:
from my_ml.linear_models.regression import LinearRegression
from my_ml.preprocessing import Data_split,KFold

In [10]:
from my_ml.trees import DecisionTree_Classification, DecisionTree_Regression,RandomForestClassification

In [11]:
import numpy as np

def confusion_matrix(y_true, y_pred):
    classes = np.unique(np.concatenate((y_true, y_pred)))
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)

    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    for t, p in zip(y_true, y_pred):
        cm[class_to_idx[t], class_to_idx[p]] += 1

    return cm


### Decision Trees

In [ ]:
X,y = load_spambase()

In [ ]:
(X_train,y_train),(X_test,y_test),(X_val,y_val) = Data_split(X,y,[0.5,0.25],True,True)

In [ ]:
fold = KFold(k_split=5,shuffle=True)
splits = list(fold.split(X))

In [ ]:
X, y = fetch_california_housing(return_X_y=True)

In [ ]:
model = DecisionTree_Regression(
   threshold=0.01,
   max_depth=5,
   min_samples_split=80,
   min_samples_leaf=40,
)


In [ ]:
model.fit(X,y)

In [ ]:
y_pred = model.predict(X)

### PCA test

In [ ]:
from my_ml.decomposition import PCA,KPCA

In [ ]:
X,y = load_spambase()

In [ ]:
decomp = KPCA(n_components=2,gamma=1.0)
decomp.fit(X)

In [ ]:
X_trans = decomp.transform(X)

In [ ]:
X_trans

array([[4.97802848e-06, 4.97938295e-06],
       [3.24702633e-06, 3.22263730e-06],
       [3.24626258e-06, 3.22186819e-06],
       ...,
       [3.24894779e-06, 3.22457240e-06],
       [3.50018160e-06, 3.47790223e-06],
       [3.24672763e-06, 3.22233660e-06]], shape=(4601, 2))

### tensor

In [1]:
import numpy as np
import my_ml
from my_ml.mytorch import tensor

In [2]:
# ---------------- Numerical gradient helper ---------------- #

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x.data)
    it = np.nditer(x.data, flags=['multi_index'], op_flags=['readwrite'])

    while not it.finished:
        idx = it.multi_index

        old = x.data[idx]

        x.data[idx] = old + eps
        f_pos = f().data.sum()

        x.data[idx] = old - eps
        f_neg = f().data.sum()

        x.data[idx] = old
        grad[idx] = (f_pos - f_neg) / (2 * eps)

        it.iternext()
    return grad


# ---------------- Test 1: Basic Ops ---------------- #

def test_basic_ops(tensor):
    print("\n=== TEST 1: BASIC OPS ===")

    a = tensor(np.random.randn(3, 4))
    b = tensor(np.random.randn(3, 4))
    c = tensor(np.random.randn(1, 4))   # broadcasting

    out = (a + b) * a
    out = out.relu() + c.sigmoid()

    s = out.data.sum()
    out.backward()

    print("Output sum:", s)
    print("Grad(a):", a.grad)
    print("Grad(b):", b.grad)
    print("Grad(c):", c.grad)


# ---------------- Test 2: Matmul + activations ---------------- #

def test_matmul(tensor):
    print("\n=== TEST 2: MATMUL ===")

    X = tensor(np.random.randn(5, 10))
    W = tensor(np.random.randn(10, 6))
    b = tensor(np.random.randn(1, 6))

    out = X @ W
    out = out + b
    out = out.sigmoid().relu()

    s = out.data.sum()
    out.backward()

    print("Loss:", s)
    print("Grad(W):", W.grad)
    print("Grad(b):", b.grad)
    print("Grad(X):", X.grad)


# ---------------- Test 3: Large MLP ---------------- #

def test_large_mlp(tensor):
    print("\n=== TEST 3: 3-LAYER MLP ===")

    np.random.seed(0)

    X = tensor(np.random.randn(32, 20))
    y = tensor(np.random.randn(32, 10))

    W1 = tensor(np.random.randn(20, 64))
    b1 = tensor(np.random.randn(1, 64))

    W2 = tensor(np.random.randn(64, 64))
    b2 = tensor(np.random.randn(1, 64))

    W3 = tensor(np.random.randn(64, 10))
    b3 = tensor(np.random.randn(1, 10))

    # forward
    h1 = (X @ W1 + b1).relu()
    h2 = (h1 @ W2 + b2).sigmoid()
    out = h2 @ W3 + b3

    loss = ((out - y) * (out - y)).data.mean()

    (out - y).backward()

    print("LOSS:", loss)
    print("Grad(W1) sum:", W1.grad.sum())
    print("Grad(W2) sum:", W2.grad.sum())
    print("Grad(W3) sum:", W3.grad.sum())
    print("Grad(X)  sum:", X.grad.sum())


# ---------------- Test 4: Gradient check ---------------- #

def test_gradcheck(tensor):
    print("\n=== TEST 4: GRAD CHECK ===")

    np.random.seed(1)
    x = tensor(np.random.randn(4, 5))

    def f():
        return (x * x + x).sigmoid().relu()

    y = f()
    y.backward()

    grad_analytical = x.grad.copy()
    grad_numeric = numerical_grad(f, x)

    diff = np.abs(grad_analytical - grad_numeric).mean()

    print("Mean absolute gradient diff:", diff)
    print("Should be < 1e-4")


# ---------------- RUN ALL ---------------- #

def run_all_tests(tensor):
    test_basic_ops(tensor)
    test_matmul(tensor)
    test_large_mlp(tensor)
    test_gradcheck(tensor)


In [3]:
run_all_tests(tensor)


=== TEST 1: BASIC OPS ===
[tensor(
  data=
    [[ 2.12255658 -0.5085971   0.22243685  0.69617013]
     [-1.90207466 -0.76306533  1.07837934  2.00478838]
     [ 0.05534517  0.03593094 -1.07174164  1.17982865]],
  grad=
    [[0. 0. 0. 0.]
     [0. 0. 0. 0.]
     [0. 0. 0. 0.]]
), tensor(
  data=
    [[ 1.04420876 -1.69493245  0.04450733 -2.18459316]
     [ 0.02597263 -0.33791874 -0.92726759  0.75656521]
     [ 0.77675447  0.65022323  0.98046972 -0.49538325]],
  grad=
    [[0. 0. 0. 0.]
     [0. 0. 0. 0.]
     [0. 0. 0. 0.]]
), tensor(
  data=
    [[ 3.16676534 -2.20352955  0.26694418 -1.48842303]
     [-1.87610203 -1.10098407  0.15111176  2.76135359]
     [ 0.83209965  0.68615417 -0.09127192  0.68444539]],
  grad=
    [[0. 0. 0. 0.]
     [0. 0. 0. 0.]
     [0. 0. 0. 0.]]
), tensor(
  data=
    [[ 6.72163861  1.12070875  0.05937822 -1.03619565]
     [ 3.56848613  0.84012277  0.1629558   5.53592958]
     [ 0.0460527   0.02465416  0.09781992  0.80752828]],
  grad=
    [[0. 0. 0. 0.]
     [

In [1]:
import numpy as np
import my_ml
from my_ml.mytorch import tensor

a = tensor(np.random.randn(3, 4))
b = tensor(np.random.randn(3, 4))
c = tensor(np.random.randn(1, 4))   # broadcasting

out = (a + b) * a
out = out.relu() + c.sigmoid()

s = out.power()

In [2]:
s.grad = 10
s.backward()

[tensor(
  data=
    [[ 0.43185398  0.42838629  0.34890953 -1.04801869]
     [-0.53615897 -0.56327473  1.04056233 -1.27079664]
     [ 1.55132017 -0.94994383  0.39189659  0.30200219]],
  grad=
    [[  29.50763927    0.            0.         -107.1697847 ]
     [-143.65548887  -38.09084285   78.88884968  -91.16870633]
     [ 202.8652414     0.            5.94993005    0.        ]]
), tensor(
  data=
    [[ 0.86732842 -0.9413192  -1.7962764  -0.38669967]
     [-2.58430514 -0.26954227  0.30028729  0.2325131 ]
     [ 0.22552248  1.36183475 -0.03897033 -1.56749613]],
  grad=
    [[  7.36148103   0.           0.         -45.2387566 ]
     [-21.06374569 -15.36833775  34.47062796 -50.17447496]
     [ 94.55935823   0.           3.13062001   0.        ]]
), tensor(
  data=
    [[ 1.29918241 -0.51293291 -1.44736687 -1.43471836]
     [-3.12046411 -0.832817    1.34084961 -1.03828354]
     [ 1.77684265  0.41189092  0.35292625 -1.26549394]],
  grad=
    [[  7.36148103   0.           0.         -45.238

In [3]:
a = tensor(np.random.randn(3, 4))

In [4]:
a = a*a
a = a/4
a = a.sum()


In [5]:
a.grad = 10

In [6]:
a

tensor(
  data=
    3.402599820096221,
  grad=
    10
)

In [7]:
a.backward()

[tensor(
  data=
    [[-0.03378935  1.57064226  1.23604614 -0.61622539]
     [-0.80082389  1.59764002  0.48444345  0.70057052]
     [ 2.17144564 -0.27932307  0.4559386  -0.56076917]],
  grad=
    [[-0.16894676  7.85321131  6.18023069 -3.08112697]
     [-4.00411946  7.98820011  2.42221725  3.50285261]
     [10.85722818 -1.39661534  2.27969301 -2.80384583]]
), tensor(
  data=
    [[1.14172036e-03 2.46691712e+00 1.52781006e+00 3.79733736e-01]
     [6.41318907e-01 2.55245364e+00 2.34685457e-01 4.90799057e-01]
     [4.71517615e+00 7.80213762e-02 2.07880008e-01 3.14462057e-01]],
  grad=
    [[2.5 2.5 2.5 2.5]
     [2.5 2.5 2.5 2.5]
     [2.5 2.5 2.5 2.5]]
), tensor(
  data=
    0.25,
  grad=
    136.10399280384885
), tensor(
  data=
    [[2.85430089e-04 6.16729279e-01 3.81952514e-01 9.49334341e-02]
     [1.60329727e-01 6.38113410e-01 5.86713643e-02 1.22699764e-01]
     [1.17879404e+00 1.95053440e-02 5.19700020e-02 7.86155142e-02]],
  grad=
    [[10. 10. 10. 10.]
     [10. 10. 10. 10.]
     [

### Module and Linear layer

In [23]:
from my_ml.mytorch.nn.module import Module
from my_ml.mytorch.nn.linear import Linear
from my_ml.mytorch.nn.activations import ReLU

class MLP(Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()

        layers = []
        dims = [input_dim] + hidden_dims

        for i in range(len(hidden_dims)):
            layers.append(Linear(dims[i], dims[i + 1]))
            layers.append(ReLU())

        layers.append(Linear(dims[-1], output_dim))

        self.layers = layers
        self.dims = dims

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return x
    
    def backward(self):
        return self.out.backward()


In [24]:
model = MLP(
    input_dim=2,
    hidden_dims=[16,16],
    output_dim=16
)


In [25]:
for layer in model.layers:
    print(layer)

Linear(in_features=2, out_features=16, bias=True)
ReLU()
Linear(in_features=16, out_features=16, bias=True)
ReLU()
Linear(in_features=16, out_features=16, bias=True)


In [26]:
y_pred = model.forward([1.0, 2.0])

In [37]:
model.backward()

[tensor(
  data=
    [1. 2.],
  grad=
    [0. 0.]
), tensor(
  data=
    [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
     [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]],
  grad=
    [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
     [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
), tensor(
  data=
    [3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
), tensor(
  data=
    [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
), tensor(
  data=
    [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
), tensor(
  data=
    [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
), tensor(
  data=
    [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
     [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
     [1. 1. 1. 1. 1. 1. 1. 

In [36]:
y_pred

tensor(
  data=
    [1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041.
     1041. 1041. 1041. 1041.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
)

### Module and Linear and back prop

In [1]:
import numpy as np
from my_ml.mytorch.nn.module import Module
from my_ml.mytorch.nn.linear import Linear
from my_ml.mytorch.nn.activations import ReLU
from my_ml.mytorch.nn.loss import MSE

class MLP(Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()

        layers = []
        dims = [input_dim] + hidden_dims

        for i in range(len(hidden_dims)):
            layers.append(Linear(dims[i], dims[i + 1]))
            layers.append(ReLU())

        layers.append(Linear(dims[-1], output_dim))

        self.layers = layers
        self.dims = dims

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return x
    
    def backward(self):
        return self.out.backward()


In [2]:
y_true = np.array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [3]:
model = MLP(
    input_dim=2,
    hidden_dims=[16,16],
    output_dim=16
)


In [4]:
for layer in model.layers:
    print(layer)

Linear(in_features=2, out_features=16, bias=True)
ReLU()
Linear(in_features=16, out_features=16, bias=True)
ReLU()
Linear(in_features=16, out_features=16, bias=True)


In [5]:
y_pred = model.forward([1.0, 2.0])

In [6]:
y_pred

tensor(
  data=
    [1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041. 1041.
     1041. 1041. 1041. 1041.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
)

In [7]:
loss = MSE()
loss(y_true,y_pred)

tensor(
  data=
    [67600. 67600. 67600. 67600. 67600. 67600. 67600. 67600. 67600. 67600.
     67600. 67600. 67600. 67600. 67600. 67600.],
  grad=
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
)

In [8]:
loss.backward()

[tensor(
  data=
    [1. 2.],
  grad=
    [532480. 532480.]
), tensor(
  data=
    [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
     [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]],
  grad=
    [[33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280.
      33280. 33280. 33280. 33280. 33280. 33280.]
     [66560. 66560. 66560. 66560. 66560. 66560. 66560. 66560. 66560. 66560.
      66560. 66560. 66560. 66560. 66560. 66560.]]
), tensor(
  data=
    [3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.],
  grad=
    [33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280.
     33280. 33280. 33280. 33280. 33280. 33280.]
), tensor(
  data=
    [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.],
  grad=
    [33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280. 33280.
     33280. 33280. 33280. 33280. 33280. 33280.]
), tensor(
  data=
    [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.],
  grad=
    [33280. 33280. 33280. 33280. 33280. 33280. 33280. 3328

In [9]:
loss.backward()

[tensor(
  data=
    [1. 2.],
  grad=
    [6389760. 6389760.]
), tensor(
  data=
    [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
     [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]],
  grad=
    [[399360. 399360. 399360. 399360. 399360. 399360. 399360. 399360. 399360.
      399360. 399360. 399360. 399360. 399360. 399360. 399360.]
     [798720. 798720. 798720. 798720. 798720. 798720. 798720. 798720. 798720.
      798720. 798720. 798720. 798720. 798720. 798720. 798720.]]
), tensor(
  data=
    [3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.],
  grad=
    [366080. 366080. 366080. 366080. 366080. 366080. 366080. 366080. 366080.
     366080. 366080. 366080. 366080. 366080. 366080. 366080.]
), tensor(
  data=
    [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.],
  grad=
    [366080. 366080. 366080. 366080. 366080. 366080. 366080. 366080. 366080.
     366080. 366080. 366080. 366080. 366080. 366080. 366080.]
), tensor(
  data=
    [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.],
